# 12 - Run the DNABERT2 representation sensitivity grid

This notebook bundles the model-strategy experiments. It creates one 64-window table, extracts window-level DNABERT2 embeddings once, and evaluates 18 strategies: 3 window caps x 2 aggregations x 3 classifiers. Each strategy is evaluated across the same 10 lineage-aware partitions.


## 1. Configuration and storage estimate

The largest new artifact is the compressed window-level embedding archive. Expect roughly 40 MB, not gigabytes. DNABERT2 inference is the slow stage and may take approximately 15-25 minutes on Apple MPS.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
print("Project root:", PROJECT_ROOT)

MANIFEST = PROJECT_ROOT / "data" / "manifests" / "isolate_manifest_public_assemblies.csv"
WINDOWS = PROJECT_ROOT / "data" / "processed" / "dnabert2_full_64_sampled_windows.csv"
MEAN_EMBEDDINGS = PROJECT_ROOT / "data" / "processed" / "dnabert2_full_64_mean_embeddings.npz"
WINDOW_EMBEDDINGS = PROJECT_ROOT / "data" / "processed" / "dnabert2_full_64_window_embeddings.npz"
REPORT = PROJECT_ROOT / "reports" / "dnabert2_strategy_evaluation.json"
LOCAL_MODEL = PROJECT_ROOT / "data" / "external" / "dnabert2_pytorch_fallback"
BUILD_64_WINDOWS = True
EMBED_64_WINDOWS = True
RUN_STRATEGY_GRID = True


## 2. Build evenly spaced 64-window input

This remains compact and reuses the existing assemblies. Completed artifacts are reused automatically, so interrupted runs can resume with **Run All**.


In [ ]:
command = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "build_sampled_windows.py"),
    "--manifest", str(MANIFEST), "--output", str(WINDOWS),
    "--maximum-windows-per-isolate", "64",
]
print(" ".join(command))
if BUILD_64_WINDOWS and not WINDOWS.exists():
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
if WINDOWS.exists():
    print("Window table MB:", round(WINDOWS.stat().st_size / 1_000_000, 2))
else:
    print("Window table is missing. Set BUILD_64_WINDOWS = True and rerun this cell.")


## 3. Extract window-level DNABERT2 embeddings once

The local PyTorch-attention fallback prepared earlier is reused. Existing embeddings are not recomputed.


In [ ]:
command = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "embed_dnabert2.py"),
    "--windows", str(WINDOWS), "--model", str(LOCAL_MODEL),
    "--maximum-windows-per-isolate", "64", "--output", str(MEAN_EMBEDDINGS),
    "--window-output", str(WINDOW_EMBEDDINGS),
]
print(" ".join(command))
if EMBED_64_WINDOWS and not WINDOW_EMBEDDINGS.exists():
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
if WINDOW_EMBEDDINGS.exists():
    print("Window embedding archive MB:", round(WINDOW_EMBEDDINGS.stat().st_size / 1_000_000, 2))
else:
    print("Window embeddings are missing. Build windows first, then set EMBED_64_WINDOWS = True.")


## 4. Evaluate 18 representation and classifier strategies

This stage should be much faster than transformer inference. Existing reports are reused.


In [ ]:
command = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "evaluate_dnabert2_strategies.py"),
    "--window-embeddings", str(WINDOW_EMBEDDINGS), "--manifest", str(MANIFEST),
    "--output", str(REPORT),
]
print(" ".join(command))
if RUN_STRATEGY_GRID and WINDOW_EMBEDDINGS.exists() and not REPORT.exists():
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
if REPORT.exists():
    report = json.loads(REPORT.read_text())
    summary = pd.DataFrame(report["summary"]).sort_values("roc_auc_mean", ascending=False)
    display(summary.round(3))
else:
    print("Strategy report is missing. Complete the window-embedding stage above, then rerun this cell.")


## 5. Decision

Prefer a strategy only if its improvement is reasonably consistent across lineage-aware folds. Small differences should be treated as uncertainty, not as a reason to add model complexity.
